# 02 — Label Audit

Run Isolation Forest as an unsupervised baseline and compare against the
provided labels. Investigate disagreements: are any 'legitimate' rows
actually undetected fraud?

In [3]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from fraud.data_loader import load_raw

df = load_raw()

In [4]:
from sklearn.ensemble import IsolationForest

X = df.drop(columns=["Class"])
iso = IsolationForest(contamination=0.0017, random_state=42)
df["iso_pred"] = iso.fit_predict(X)

df["iso_pred"].value_counts()

iso_pred
 1    284322
-1       485
Name: count, dtype: int64

In [5]:
pd.crosstab(df["Class"], df["iso_pred"],
            rownames=["Actual"], colnames=["IsoForest"])

IsoForest,-1,1
Actual,,
0,360,283955
1,125,367


In [6]:
suspects = df[(df["Class"] == 0) & (df["iso_pred"] == -1)]
suspects[["Amount", "V14", "V17", "V12"]].describe()

,Amount,V14,V17,V12
count,360.000000,360.000000,360.000000,360.000000
mean,2518.825750,-0.492869,-1.135262,-0.477522
std,3078.055851,4.017159,4.069672,3.734656
min,0.000000,-18.392091,-16.825344,-15.117889
25%,164.045000,-2.262538,-1.394822,-1.111392
50%,1441.060000,-0.018196,-0.476647,0.146963
75%,4283.790000,1.708318,0.656313,1.204692
max,25691.160000,10.526766,9.207059,7.848392
